In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Подключаем пути
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Импортируем наши модули
import src.llm_platform.config as config
from src.llm_platform.data_foundry.llm_client import LLMClient
from src.llm_platform.data_foundry.schemas import SFTPair

print("Среда настроена. Ключ API загружен:", bool(config.OPENROUTER_API_KEY))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Среда настроена. Ключ API загружен: True


In [4]:
# Инициализируем клиента с бесплатной, но умной моделью
client = LLMClient(model_name="openai/gpt-oss-120b")

# Имитируем промты, которые потом будет генерировать наш пайплайн
system_prompt = (
    "You are an expert AI data generator. "
    "Your task is to read the provided text chunk and extract one highly specific, "
    "complex question and its detailed answer. Ensure the answer is fully contained "
    "within the source text."
)

user_prompt_chunk = (
    "В 2026 году компания NVIDIA представила архитектуру Rubin, которая "
    "значительно ускорила инференс Mixture-of-Experts моделей благодаря "
    "новым тензорным ядрам 5-го поколения и памяти HBM4 с пропускной "
    "способностью до 8 ТБ/с."
)

print(f"Отправляем запрос к {client.model_name}...")

try:
    # Делаем реальный сетевой запрос
    result = await client.generate_structured_data(
        system_prompt=system_prompt,
        user_prompt=user_prompt_chunk,
        response_model=SFTPair
    )
    
    print("\nУспех! Модель вернула валидный Pydantic-объект:")
    # Выводим результат в красивом JSON-формате
    print(result.model_dump_json(indent=2))

except Exception as e:
    print(f"\nОшибка во время генерации: {e}")

2026-05-31 20:32:18 [INFO] (src.llm_platform.data_foundry.llm_client): LLMClient initialized with model: openai/gpt-oss-120b


Отправляем запрос к openai/gpt-oss-120b...


2026-05-31 20:32:19 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"



Успех! Модель вернула валидный Pydantic-объект:
{
  "pair_id": "pair_001",
  "source_chunk_id": "chunk_001",
  "messages": [
    {
      "role": "user",
      "content": "What specific hardware innovations did NVIDIA introduce with the Rubin architecture in 2026 that accelerate inference for Mixture-of-Experts models?"
    },
    {
      "role": "assistant",
      "content": "The Rubin architecture, announced by NVIDIA in 2026, accelerates inference of Mixture-of-Experts models through two key hardware innovations: new 5th‑generation tensor cores and HBM4 memory with a bandwidth of up to 8 TB/s."
    }
  ],
  "is_evolved": false
}
